In [1]:
import os
import random
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

Préparer les images

In [34]:
def process_dataset_pipeline(source_dir, output_dir):
    """
    Processes a nested dataset of images:
    1. Applies grayscale, bilateral filter, and Canny edge detection.
    2. Finds square contours to extract the Region of Interest (ROI).
    3. Normalizes ROI to 40x40, converts to grayscale, and applies a blur.
    4. Saves to output folder keeping original folder structure.
    """
    source_path = Path(source_dir)
    output_path = Path(output_dir)
    
    image_extensions = {'.jpg'}
    processed_records = []  # Stores (file_path, label) for visualization
    
    if not source_path.exists():
        print(f"Error: Source directory '{source_dir}' does not exist.")
        return processed_records

    print("Processing images...")
    
    for type_dir in source_path.iterdir():
        if not type_dir.is_dir():
            continue
            
        for subtype_dir in type_dir.iterdir():
            if not subtype_dir.is_dir():
                continue
            
            # Create destination folder structure
            dest_subtype_dir = output_path / type_dir.name / subtype_dir.name
            dest_subtype_dir.mkdir(parents=True, exist_ok=True)
            
            images = [f for f in subtype_dir.iterdir() if f.is_file() and f.suffix.lower() in image_extensions]
            
            for img_path in images:
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                
                # --- Step 1: Image Preparation ---
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                bilateral = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)
                edges = cv2.Canny(bilateral, 50, 150)
                
                # --- Step 2: ROI Extraction (Square contours) ---
                contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                best_roi = None
                max_area = 0
                
                for cnt in contours:
                    area = cv2.contourArea(cnt)
                    if area > 100:  # Minimum area threshold
                        perimeter = cv2.arcLength(cnt, True)
                        approx = cv2.approxPolyDP(cnt, 0.03 * perimeter, True)
                        
                        # Look for square/rectangular shapes (4 vertices)
                        if len(approx) == 4:
                            x, y, cw, ch = cv2.boundingRect(approx)
                            aspect_ratio = float(cw) / ch if ch > 0 else 0
                            
                            # Filter for roughly square aspect ratio
                            if 0.8 <= aspect_ratio <= 1.2 and area > max_area:
                                max_area = area
                                best_roi = img[y:y+ch, x:x+cw]
                
                # Fallback if no square contour is found
                if best_roi is None or best_roi.size == 0:
                    best_roi = img
                
                # --- Step 3: Extracted Region Preparation ---
                # 1. Resize/Normalize dimensions to 40 x 40
                roi_resized = cv2.resize(best_roi, (40, 40), interpolation=cv2.INTER_AREA)
                
                # 2. Convert to grayscale
                if len(roi_resized.shape) == 3:
                    roi_gray = cv2.cvtColor(roi_resized, cv2.COLOR_BGR2GRAY)
                else:
                    roi_gray = roi_resized
                    
                # 3. Apply blur to smooth details
                roi_blurred = cv2.GaussianBlur(roi_gray, (3, 3), 0)
                
                # Save processed image
                dest_file_path = dest_subtype_dir / img_path.name
                cv2.imwrite(str(dest_file_path), roi_blurred)
                
                # Keep record for visualization (label is the sub-folder name)
                processed_records.append((dest_file_path, subtype_dir.name))
                
    print(f"Pipeline complete! Processed {len(processed_records)} images.")
    return processed_records

In [ ]:
SOURCE_DATASET = r"Ensemble_entrainement"
OUTPUT_DATASET = r"Ensemble_processed"

records = process_dataset_pipeline(SOURCE_DATASET, OUTPUT_DATASET)

Processing images...
Pipeline complete! Processed 3860 images.


Vérifier le pré-processing des images

In [ ]:
def plot_random_samples(processed_records, sample_size=16):
    """Picks 16 random processed images and displays them in a 4x4 grid with their labels."""
    if not processed_records:
        print("No processed records available to display.")
        return
        
    sample_size = min(16, len(processed_records))
    selected_samples = random.sample(processed_records, sample_size)
    
    plt.figure(figsize=(12, 12))
    for i, (img_path, label) in enumerate(selected_samples):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        
        plt.subplot(4, 4, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"Label: {label}", fontsize=10)
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# Run the visualization
plot_random_samples(records, sample_size=16)